# LAB-HW-03 — Generate and Program the First Bitstream

**One problem today: turn a tiny RTL design into a configuration actually running in KV260 PL.**

Prerequisites: LAB-HW-00~02 passed. Do not learn Linux, AXI, or redesign the neuron today.

**Project Trace:** RMD-012A · T-HW-003/T-HW-011

## 1. What will the FPGA do?

The course-provided `kv260_marker_top` has one behavior:

```systemverilog
bank45_gpio = 5'b10101;
```

No clock, reset, or host communication.

That keeps today's failure domain limited to **build / implementation / bitstream / JTAG programming / physical mapping**.

<svg xmlns="http://www.w3.org/2000/svg" width="820" height="250" viewBox="0 0 820 250" role="img" aria-label="LAB-HW-03 first bitstream flow">
  <rect x="20" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="85" y="110" text-anchor="middle" font-size="15">marker RTL</text>
  <text x="85" y="133" text-anchor="middle" font-size="12">5'b10101</text>
  <rect x="190" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="255" y="110" text-anchor="middle" font-size="15">Vivado</text>
  <text x="255" y="133" text-anchor="middle" font-size="12">synth + impl</text>
  <rect x="360" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="425" y="110" text-anchor="middle" font-size="15">bitstream</text>
  <text x="425" y="133" text-anchor="middle" font-size="12">.bit + reports</text>
  <rect x="530" y="80" width="130" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="595" y="110" text-anchor="middle" font-size="15">JTAG program</text>
  <text x="595" y="133" text-anchor="middle" font-size="12">xck26*</text>
  <rect x="700" y="80" width="100" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="750" y="110" text-anchor="middle" font-size="14">Bank45</text>
  <text x="750" y="133" text-anchor="middle" font-size="12">visible marker</text>
  <path d="M150 115 L190 115 M320 115 L360 115 M490 115 L530 115 M660 115 L700 115" stroke="#333" stroke-width="2"/>
  <polygon points="190,115 180,110 180,120" fill="#333"/><polygon points="360,115 350,110 350,120" fill="#333"/>
  <polygon points="530,115 520,110 520,120" fill="#333"/><polygon points="700,115 690,110 690,120" fill="#333"/>
</svg>

## 2. Why these five outputs?

AMD/Xilinx Board Store data describes KV260 carrier `bank45_gpio` as a 5-bit LED-class GPIO output. The K26 SOM pin map gives the package pins:

| bit | SOM240 | package pin |
|---|---|---|
| 0 | D18 | J11 |
| 1 | B17 | J10 |
| 2 | B18 | K13 |
| 3 | A15 | F11 |
| 4 | C24 | A12 |

The course records this mapping in `boards/kv260/constraints/bank45_gpio.xdc`.

**Do not edit the XDC today.** Treat it as a reviewed board adapter. LAB-HW-04 explains `PACKAGE_PIN` and `IOSTANDARD`.

## 3. Build: RTL → bitstream

From the repository root:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-03-build.log \
  -source boards/kv260/scripts/build_lab03_marker.tcl
```

The script targets `xck26-sfvc784-2LV-c` and runs synthesis, optimization, placement, routing, and bitstream generation.

Expected outputs:

```text
build/kv260/lab-hw-03/kv260_marker_top.bit
build/kv260/lab-hw-03/timing_summary.rpt
build/kv260/lab-hw-03/utilization.rpt
```

`STATUS=PASS` proves only that the build helper completed. It is not a physical-board PASS.

## 4. Give the bitstream an unambiguous identity

Linux/macOS shell:

```bash
sha256sum build/kv260/lab-hw-03/kv260_marker_top.bit
```

Windows PowerShell:

```powershell
Get-FileHash build/kv260/lab-hw-03/kv260_marker_top.bit -Algorithm SHA256
```

Record the SHA-256 in the evidence manifest. From now on, “this bitstream” means the artifact identified by that hash, not merely a same-named file.

## 5. Program: send only the built .bit to the real XCK26

Keep the LAB-HW-02 power and J4 JTAG connection.

Run:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-03-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-03/kv260_marker_top.bit
```

The helper rejects:

- missing bitstream;
- a non-`.bit` argument;
- a JTAG chain without an `xck26*` device;
- a `program_hw_devices` failure.

A success log should contain `KV260_FPGA_DEVICE=xck26...` and `STATUS=PASS`.

## 6. Expected Evidence

After programming, observe the stable state on the Bank 45 LED-class outputs.

A deliberate evidence boundary remains: **until the course completes a real-KV260 dry run, it does not invent a tested silkscreen LED name or visible polarity.** Board Store data supports the interface and pin mapping; this physical run records the actual visible designator/polarity.

T-HW-003 requires at least:

- `lab-hw-03-build.log`;
- `timing_summary.rpt` and `utilization.rpt`;
- bitstream SHA-256;
- `lab-hw-03-program.log`;
- `xck26*` target identification;
- real-board marker observation/photo;
- carrier revision + Git commit.

### Save Evidence

Copy `boards/kv260/evidence/manifest.example.json` to a local LAB-HW-03 evidence file and fill those fields.

## 7. Why DS34 is not today's primary answer

UG1089 defines DS34 as lit when the **PS successfully loaded a PL design**.

Today's path is development host → JTAG → direct PL programming.

Therefore the primary oracle is:

**Vivado programming result + XCK26 identity + the design's own Bank 45 output.**

Do not mark T-HW-003 PASS from DS34 alone.

## 8. If it does not work

Debug by layer:

1. **build fails** → inspect synthesis/implementation errors first;
2. **no bitstream** → do not proceed to programming;
3. **no XCK26** → return to LAB-HW-02 and check J12/J4/driver/JTAG;
4. **programming fails** → retain the program log; do not edit neuron RTL;
5. **program succeeds but no board-visible change** → inspect carrier revision, XDC, and actual board output. That is a physical-mapping problem, not a reason to repeatedly click Program Device.

## 9. Human Check

Explain:

1. How do synthesis, implementation, bitstream generation, and programming differ?
2. Why does a `.bit` artifact need a SHA-256?
3. Why does an RTL port named `bank45_gpio` not intrinsically know package pin J11?
4. Why can DS34 not by itself prove today's direct-JTAG programming path?
5. Why did this lab intentionally avoid clock, reset, AXI, and neuron RTL?

## 10. Official basis

- AMD UG1089 — Interfaces / J4 / reset / DS34  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Interfaces
- AMD/Xilinx Board Store — KV260 carrier `bank45_gpio`  
  https://github.com/Xilinx/XilinxBoardStore/blob/master/boards/Xilinx/kv260_carrier/1.3/board.xml
- AMD/Xilinx Board Store — K26 SOM package pin map  
  https://github.com/Xilinx/XilinxBoardStore/blob/master/boards/Xilinx/kv260_som/1.4/part0_pins.xml
- AMD DS987 — K26 SOM signal descriptions  
  https://docs.amd.com/r/en-US/ds987-k26-som/SOM240_1-Signal-Names-and-Descriptions